<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Unsupervised learning

**[Unsupervised learning](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/)** · Machine Learning: From Problem to Reliable Model · Module 1, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** group examples that have no labels with k-means, compare the groups with what a person expects, and see why a person must judge and name the groups.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Features, labels, `X` and `y`, from [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/). |
| **You do not need** | A local project. The setup below downloads the data. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M01-L03-unsupervised-learning/unsupervised-learning-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The same swatches, without names

In the last lesson, every swatch had a name card: the label. Now imagine a box of the same swatches with no cards. Nobody tells you the right groups. You can still put similar colours together. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/#sorting-a-box-of-buttons) tells the full story.

Run the next cell. It makes the 15 swatches again, but keeps the names in a separate column `hidden_names`. The method sees only `X`. You use the names only at the end, to compare.

In [ ]:
swatches = pd.DataFrame(
    [("red", 220, 40, 40), ("red", 190, 25, 35), ("red", 235, 70, 60),
     ("orange", 245, 135, 35), ("orange", 235, 115, 25), ("orange", 250, 155, 60),
     ("yellow", 250, 220, 50), ("yellow", 240, 205, 40), ("yellow", 255, 235, 100),
     ("green", 50, 165, 70), ("green", 90, 190, 80), ("green", 30, 130, 55),
     ("blue", 40, 100, 200), ("blue", 80, 140, 225), ("blue", 30, 70, 170)],
    columns=["hidden_names", "red", "green", "blue"],
)
X = swatches[["red", "green", "blue"]]  # the features: no label this time
print(X.shape)

> **Check.** You should see `(15, 3)`: 15 examples, 3 features, and no `y`.

## 2. Sort the swatches into piles with k-means

**k-means** is a method that sorts rows into `k` groups, called **clusters**. You choose `k`. It puts each row in the group whose centre is nearest, moves each centre to the middle of its group, and repeats until nothing changes. Here, `k` is 5.

There is no `y` in `fit_predict(X)`. The method learns from the features alone: that is **unsupervised learning**.

> **Predict.** Will the five piles match the five colour names?

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, n_init=10, random_state=0)
swatches["pile"] = kmeans.fit_predict(X)  # no labels here
print(swatches["pile"].tolist())

> **Check.** You should see `[1, 1, 1, 4, 4, 4, 3, 3, 3, 2, 2, 2, 0, 0, 0]`: five piles of three. The order of the rows is the order of the table: three reds, three oranges, three yellows, three greens, three blues.

Now compare the piles with the names that the method never saw. Run the next cell. Each row is a pile, and each column is a name.

In [ ]:
pd.crosstab(swatches["pile"], swatches["hidden_names"])

> **Check.** You should see a 3 in each row, and only one 3 in each column: every pile has one colour name.

The method found the same groups as a person, without a single label. It did not find the **names**: a pile is only a number. A person must look at pile 1 and say "these are the reds".

## 3. A picture of the piles

The swatches have three features, so they are points in three dimensions. Run the next cell. It draws two of the features: the amount of red light and the amount of green light. Each point has its own colour, and its pile number next to it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
rgb = X.to_numpy() / 255
ax.scatter(X["red"], X["green"], c=rgb, s=160, edgecolors="black")
for _, row in swatches.iterrows():
    ax.annotate(str(row["pile"]), (row["red"], row["green"]), xytext=(8, 4), textcoords="offset points")
ax.set_xlabel("Amount of red light (0 to 255)")
ax.set_ylabel("Amount of green light (0 to 255)")
ax.set_title("15 swatches and their pile numbers")
plt.show()

> **Check.** You should see red, orange and yellow on the right, from bottom to top, and green and blue on the left. Each colour has one pile number.

Points that are close in the picture are similar colours, and k-means put them in one pile. But green and blue are close together in this picture: the third feature, blue light, separates them. A picture of two features helps, but it can hide a difference.

## 4. Change one thing: two piles instead of five

Different people sort the same box in different ways. One person makes five piles by colour name. Another makes two: warm colours and cool colours. Neither is wrong.

> **Your turn.** Sort the swatches into 2 piles with `KMeans(n_clusters=2, n_init=10, random_state=0)`. Put the pile of each swatch in `swatches["pile2"]`. Then put the number of swatches in each pile, from the largest to the smallest, in the list `pile_sizes`. Before you run it, predict which colours go together.

In [ ]:
swatches["pile2"] = ...  # KMeans with 2 clusters, fit_predict(X)
pile_sizes = ...         # sorted(..., reverse=True) of the counts per pile
print(pile_sizes)

In [ ]:
expect_hash('pile_sizes', pile_sizes, '23dcc0a2e86b99a1')

Run the next cell to see which names are in each of the two piles.

In [ ]:
pd.crosstab(swatches["pile2"], swatches["hidden_names"])

> **Check.** In the solution, one pile has all the red, orange and yellow swatches (warm), and the other has the green and blue ones (cool).

Five piles or two? Nothing in the data says which is right. It depends on what you want to do with the piles. A painter who mixes colours may want five; a designer who chooses a warm or a cool theme may want two. **A person judges the result by its use.**

## 5. Failure case: a pile number is not a name

Tomás saves the five piles, and writes in his notes: "pile 1 = red". The next day, he runs the same method again, with another `random_state`.

> **Predict.** Is pile 1 still the reds?

In [ ]:
again = KMeans(n_clusters=5, n_init=10, random_state=1).fit_predict(X)
print("First run: ", swatches["pile"].tolist())
print("Second run:", again.tolist())
print("Reds are in pile", again[0], "now")

> **Check.** You should see the same five groups, but with other numbers: the reds are now in pile 2.

The groups are the same. Only the numbers changed. The number of a cluster is a random name tag, so never write code or notes that depend on "pile 1". Describe each group by what is in it (for example, its mean values), and give it a name that a person chose.

## 6. Larkfield: groups of customers

Grace asks: "What kinds of customers do we have?" No column answers this, so there is no label. Run the next cell. It makes one row per customer from the train tickets: the number of tickets, and how long the customer has been with Larkfield (days). Guests have no customer ID, so they are left out.

In [ ]:
train = pd.read_csv("data/train.csv", dtype={"customer_id": "string"},
                    keep_default_na=False, na_values=[""])
known = train.dropna(subset=["customer_id"])
customers = known.groupby("customer_id").agg(
    tickets=("ticket_id", "size"),
    tenure_days=("customer_tenure_days", "max"),
)
print(customers.shape)
customers.head(3)

> **Check.** You should see `(847, 2)`: 847 customers, two features, no label.

Run the next cell. It sorts the customers into 3 groups with k-means. One step comes first: `StandardScaler` puts both columns on the same scale. Without it, the days (up to about 2,600) would count much more than the tickets. [Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/) explains this step.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

grouper = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=0))
customers["group"] = grouper.fit_predict(customers[["tickets", "tenure_days"]])
customers.groupby("group").agg(
    customers=("tickets", "size"), tickets=("tickets", "mean"), tenure_days=("tenure_days", "mean")
).round(1)

> **Check.** You should see three groups. Group 0 has 401 customers (about 16 tickets, about 1,883 days). Group 1 has 384 customers (about 11 tickets, about 595 days). Group 2 has 62 customers (about 102 tickets each).

The method gives three numbers. Grace gives the names: "long-time customers", "newer customers" and "a few very busy accounts". She also decides if the groups are useful. For example, the 62 busy accounts may need an account manager. No metric can tell her that: there is no right answer to compare with.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this notebook, k-means sorted 15 swatches without labels into five piles that match the colour names. When you asked for two piles, it made a warm one and a cool one. The pile numbers changed when the random start changed; the groups did not. Larkfield's 847 customers fell into three groups that only Grace can name and judge. Learning from features without labels is **unsupervised learning**, and **clustering** is its most common task.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Choose the learning task](https://learning.nextia-ai.com/courses/ml/m01/choose-the-learning-task/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m01/unsupervised-learning/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.